# Notebook 05 — Comparación: BoW vs Word2Vec vs BETO
**Proyecto 2 | Análisis Semántico de Letras Musicales**  
**Curso:** Minería de Textos — CUC  

---
## Objetivo
Comparar cuantitativamente las tres representaciones evaluando:
1. Clasificación de género (Logistic Regression)
2. Clustering (K-Means + Silhouette Score)
3. Visualización t-SNE comparativa


## 0. Dependencias e imports

In [ ]:
!pip install -q scikit-learn matplotlib seaborn gensim transformers torch pandas numpy pymongo python-dotenv
import sys
sys.path.append("..")
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

print("Imports listos")

## 1. Cargar corpus y modelos

In [ ]:
from src.entities.consultar_base_datos import consultar_base_datos

cargador = consultar_base_datos()
cargador.cargar_por_generos(["pop","alternative pop","hip hop","alternative rock","dance pop", "rock"])
df = cargador.df

# Usar una muestra equilibrada por género para comparación justa
MIN_POR_GENERO = 100
grupos = []
for genero, grp in df.groupby("genero"):
    n = min(len(grp), MIN_POR_GENERO)
    grupos.append(grp.sample(n, random_state=42))

df_eq = pd.concat(grupos).reset_index(drop=True)
print(f"Dataset balanceado: {len(df_eq)} canciones")
print(df_eq["genero"].value_counts().to_string())

In [ ]:
# ── Representación 1: TF-IDF ──
print("Construyendo TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, min_df=2, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df_eq["letra"].fillna("")).toarray()
print(f"TF-IDF: {X_tfidf.shape}")

In [ ]:
# ── Representación 2: Word2Vec promedio ──
print("Cargando Word2Vec...")
from gensim.models import Word2Vec
from src.embeddings.embeddings_w2v import EntrenadorWord2Vec, calcular_vector_promedio

entrenador = EntrenadorWord2Vec(df_eq, col_letra="letra")
entrenador.entrenar_skip_gram(vector_size=100, epochs=10)
wv = entrenador.modelo_sg.wv

X_w2v = np.array([
    calcular_vector_promedio(str(letra), wv) or [0.0]*100
    for letra in df_eq["letra"].fillna("")
])
print(f"Word2Vec: {X_w2v.shape}")

In [ ]:
# ── Representación 3: BETO [CLS] ──
print("Cargando BETO (puede tardar)...")
from src.embeddings.embeddings_beto import CargadorBETO, embedding_cls

beto = CargadorBETO()
X_beto = embedding_cls(
    df_eq["letra"].fillna("").tolist(),
    beto.tokenizer, beto.model,
    batch_size=16
)
print(f"BETO: {X_beto.shape}")

In [ ]:
# Etiquetas
le = LabelEncoder()
y = le.fit_transform(df_eq["genero"])
clases = le.classes_
print(f"Clases: {clases}")

## 2. Clasificación de género

In [ ]:
def evaluar_clasificador(X, y, nombre, clf=None):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    if clf is None:
        clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    print(f"\n{'='*50}")
    print(f"Clasificador: {nombre}")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_te, y_pred, target_names=clases))
    return acc

# Evaluar las 3 representaciones
acc_bow  = evaluar_clasificador(X_tfidf, y, "TF-IDF + LogReg")
acc_w2v  = evaluar_clasificador(X_w2v,   y, "Word2Vec + LogReg")
acc_beto = evaluar_clasificador(X_beto,  y, "BETO + LogReg")

In [ ]:
# ── Visualización comparativa de accuracy ──
nombres = ["TF-IDF", "Word2Vec", "BETO"]
accuracies = [acc_bow, acc_w2v, acc_beto]
colores_barra = ["#3498db", "#e74c3c", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(nombres, accuracies, color=colores_barra, alpha=0.85, edgecolor="white", linewidth=0.5)
for barra, acc in zip(barras, accuracies):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 0.005,
            f"{acc:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_ylim(0, 1.0)
ax.set_ylabel("Accuracy (test)", fontsize=11)
ax.set_title("Clasificación de género: BoW vs Word2Vec vs BETO",
             fontsize=12, fontweight="bold")
ax.axhline(y=1/len(clases), color="gray", linestyle="--", alpha=0.5, label="Baseline aleatorio")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Clustering (K-Means + Silhouette Score)

In [ ]:
from sklearn.preprocessing import normalize

K = len(clases)
resultados_clustering = {}

for nombre, X in [("TF-IDF", X_tfidf), ("Word2Vec", X_w2v), ("BETO", X_beto)]:
    X_norm = normalize(X)
    km = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = km.fit_predict(X_norm)
    sil = silhouette_score(X_norm, labels, sample_size=min(2000, len(X_norm)))
    resultados_clustering[nombre] = {"silhouette": sil, "labels": labels}
    print(f"{nombre:10s} | Silhouette Score: {sil:.4f}")

In [ ]:
# Visualizar Silhouette Score
nombres_c = list(resultados_clustering.keys())
sil_scores = [resultados_clustering[n]["silhouette"] for n in nombres_c]

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(nombres_c, sil_scores, color=["#3498db","#e74c3c","#2ecc71"],
                alpha=0.85, edgecolor="white")
for b, s in zip(barras, sil_scores):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.002,
            f"{s:.4f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylim(0, max(sil_scores) * 1.2)
ax.set_ylabel("Silhouette Score", fontsize=11)
ax.set_title("Calidad del clustering K-Means por representación",
             fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSilhouette Score más cercano a 1.0 = clusters más cohesivos")

## 4. Visualización t-SNE comparativa

In [ ]:
from sklearn.decomposition import TruncatedSVD

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
colores_g = {g: c for g, c in zip(clases, ["#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6","#1abc9c"])}

representaciones = [
    ("TF-IDF",    X_tfidf, True),
    ("Word2Vec",  X_w2v,   False),
    ("BETO",      X_beto,  False),
]

for ax, (nombre, X, es_sparse) in zip(axes, representaciones):
    # Reducir dimensiones antes de t-SNE cuando X es muy grande
    X_red = X
    if X.shape[1] > 50:
        n_comp = min(50, X.shape[1], X.shape[0] - 1)
        if es_sparse:
            from sklearn.decomposition import TruncatedSVD
            svd = TruncatedSVD(n_components=n_comp, random_state=42)
            X_red = svd.fit_transform(X)
        else:
            from sklearn.decomposition import PCA
            pca = PCA(n_components=n_comp, random_state=42)
            X_red = pca.fit_transform(X)

    perp = min(30, len(X_red) - 1)
    tsne_m = TSNE(n_components=2, random_state=42, perplexity=perp)
    X_2d = tsne_m.fit_transform(X_red)

    for genero in clases:
        idx = np.where(le.transform([genero])[0] == y)[0]
        color = colores_g.get(genero, "#95a5a6")
        ax.scatter(X_2d[idx, 0], X_2d[idx, 1], c=color,
                   label=genero, s=25, alpha=0.6, edgecolors="none")

    ax.set_title(nombre, fontsize=13, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2)
    ax.grid(True, alpha=0.2, linestyle="--")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("t-SNE comparativo: BoW vs Word2Vec vs BETO",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Tabla resumen final

In [ ]:
import pandas as pd

resumen = pd.DataFrame({
    "Representación": ["TF-IDF (BoW)", "Word2Vec (Skip-Gram)", "BETO [CLS]"],
    "Dimensión":      [X_tfidf.shape[1], X_w2v.shape[1], X_beto.shape[1]],
    "Tipo":           ["Dispersa", "Densa estática", "Densa contextual"],
    "Accuracy (LogReg)": [f"{acc_bow:.3f}", f"{acc_w2v:.3f}", f"{acc_beto:.3f}"],
    "Silhouette Score":  [f"{resultados_clustering['TF-IDF']['silhouette']:.4f}",
                          f"{resultados_clustering['Word2Vec']['silhouette']:.4f}",
                          f"{resultados_clustering['BETO']['silhouette']:.4f}"],
})

print("=== TABLA RESUMEN COMPARATIVA ===")
print(resumen.to_string(index=False))
print()
mejor_acc = resumen.loc[resumen["Accuracy (LogReg)"].idxmax(), "Representación"]
print(f"Mejor accuracy de clasificación: {mejor_acc}")